# Open Buildings v3 para Neiva
Extrae las huellas de edificaciones del área de Neiva y las guarda como `open_buildings_neiva.csv` en su Google Drive.
Licencia de los datos: CC BY 4.0 u ODbL (Google Research). Ejecute las celdas en orden.

In [ ]:
# 1. Conecta Google Drive (pide su autorización)
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# 2. Parámetros: rectángulo de Neiva
LON_MIN, LON_MAX, LAT_MIN, LAT_MAX = -75.34, -75.22, 2.86, 3.01
TILES = 'https://openbuildings-public-dot-gweb-research.uw.r.appspot.com/public/tiles.geojson'

In [ ]:
# 3. Busca la celda S2 que cubre Neiva
import requests

idx = requests.get(TILES, timeout=60).json()
lon, lat = (LON_MIN+LON_MAX)/2, (LAT_MIN+LAT_MAX)/2
tile = next(f['properties'] for f in idx['features'] if min(p[0] for p in f['geometry']['coordinates'][0]) <= lon <= max(p[0] for p in f['geometry']['coordinates'][0]) and min(p[1] for p in f['geometry']['coordinates'][0]) <= lat <= max(p[1] for p in f['geometry']['coordinates'][0]))
print(tile)

In [ ]:
# 4. Descarga por streaming y filtra el rectángulo (pocos minutos)
import csv
import gzip
import io

salida = '/content/drive/MyDrive/open_buildings_neiva.csv'
n = 0
with requests.get(tile['tile_url'], stream=True, timeout=120) as r, open(salida, 'w', newline='', encoding='utf-8') as out:
    r.raise_for_status()
    lector = csv.DictReader(io.TextIOWrapper(gzip.GzipFile(fileobj=r.raw), encoding='utf-8'))
    w = csv.writer(out); w.writerow(['confidence', 'area_in_meters', 'geometry'])
    for f in lector:
        la, lo = float(f['latitude']), float(f['longitude'])
        if LON_MIN <= lo <= LON_MAX and LAT_MIN <= la <= LAT_MAX:
            w.writerow([f['confidence'], f['area_in_meters'], f['geometry']]); n += 1
print(n, 'huellas guardadas en', salida)

# 5. Siguiente paso
Descargue `open_buildings_neiva.csv` y cópielo a `data/raw/` del repositorio; luego corra `python -m pipeline.run`.